
-SCT213-C002-0141/2024 GACHOMO NJERU

-SCT213-C002-0063/2024 MWITA NYEHITA

-SCT213-C002-0142/2024 KARWEGA NJERU


**Objectives:**
- Build an exploratory visualization system with multiple chart types
- Visualize distributions and variable relationships
- Apply and justify visualization design principles
- Compare two distinct visualization strategies (Matplotlib vs Seaborn)
- Produce an insight analysis report from the visuals

## 0. Setup & Data Preparation
_Rebuild the processed dataset from Sprint 2 so this notebook is fully self-contained._

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Consistent random seed ─────────────────────────────────────────────────────
np.random.seed(42)

# ── Load raw dataset ───────────────────────────────────────────────────────────
url = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv'
df_raw = pd.read_csv(url)

# ── Reproduce Sprint-2 processing pipeline ─────────────────────────────────────
df = df_raw.copy()

# Feature engineering
df['tip_pct']        = (df['tip'] / df['total_bill'] * 100).round(2)
df['bill_per_person']= (df['total_bill'] / df['size']).round(2)
day_avg              = df.groupby('day')['tip'].mean().round(2).reset_index()
day_avg.columns      = ['day', 'day_avg_tip']
df                   = pd.merge(df, day_avg, on='day', how='left')
df['tip_vs_day_avg'] = (df['tip'] - df['day_avg_tip']).round(2)
df['is_weekend']     = df['day'].isin(['Sat', 'Sun']).astype(int)

def tip_category(pct):
    if pct < 10:  return 'Low (<10%)'
    elif pct < 20: return 'Average (10–20%)'
    else:          return 'Generous (>20%)'

df['tip_category']    = df['tip_pct'].apply(tip_category)
df['sex_encoded']     = (df['sex']    == 'Male').astype(int)
df['smoker_encoded']  = (df['smoker'] == 'Yes').astype(int)
df['time_encoded']    = (df['time']   == 'Dinner').astype(int)

print(f"Dataset ready  |  Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

---
## 1. Visualization Design Principles

All charts in this notebook follow five core principles drawn from Tufte (2001) and Cairo (2016):

| # | Principle | Implementation |
|---|-----------|----------------|
| 1 | **Data-ink ratio** | Spines removed; gridlines minimal or absent |
| 2 | **Colour with purpose** | Colour encodes a data dimension, not decoration |
| 3 | **Consistent labelling** | Every axis and legend is labelled; units shown |
| 4 | **Chart type appropriateness** | Chart type matched to data type and question |
| 5 | **Layered complexity** | Simple → complex; annotations added only where they add value |

In [ ]:
# ── Global style settings ──────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi'       : 120,
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'axes.grid'        : True,
    'grid.alpha'       : 0.3,
    'font.family'      : 'DejaVu Sans',
    'axes.titlesize'   : 13,
    'axes.labelsize'   : 11,
})

# Palette shared across matplotlib charts
PALETTE = ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0']
print("Global style configured.")

---
## 2. Strategy A — Matplotlib (Manual, Fine-Grained Control)

**Design rationale:** Matplotlib gives explicit control over every element.  
It is best suited for custom layouts, publication-quality figures, and situations where you need pixel-level control over annotations and legends.

### 2.1 Histogram — Distribution of Total Bill

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

ax.hist(df['total_bill'], bins=25, color='#2196F3', edgecolor='white', linewidth=0.6)
ax.axvline(df['total_bill'].mean(),   color='#FF5722', lw=1.8, ls='--', label=f"Mean  ${df['total_bill'].mean():.2f}")
ax.axvline(df['total_bill'].median(), color='#4CAF50', lw=1.8, ls=':',  label=f"Median ${df['total_bill'].median():.2f}")

ax.set_title('Distribution of Total Bill (Histogram)', fontweight='bold')
ax.set_xlabel('Total Bill ($)')
ax.set_ylabel('Frequency')
ax.legend(frameon=False)

plt.tight_layout()
plt.savefig('fig_01_histogram.png', dpi=120, bbox_inches='tight')
plt.show()
print("▶ Saved: fig_01_histogram.png")

### 2.2 Bar Chart — Average Tip by Day of Week

In [ ]:
day_order  = ['Thur', 'Fri', 'Sat', 'Sun']
day_stats  = df.groupby('day')['tip'].agg(['mean', 'sem']).reindex(day_order).reset_index()

fig, ax = plt.subplots(figsize=(8, 4))

bars = ax.bar(
    day_stats['day'], day_stats['mean'],
    color=PALETTE, edgecolor='white', linewidth=0.6,
    yerr=day_stats['sem'], capsize=4, error_kw={'elinewidth': 1.2}
)

for bar, val in zip(bars, day_stats['mean']):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.05,
            f'${val:.2f}', ha='center', va='bottom', fontsize=9)

ax.set_title('Average Tip by Day of Week (with 95 % CI)', fontweight='bold')
ax.set_xlabel('Day')
ax.set_ylabel('Average Tip ($)')
ax.set_ylim(0, day_stats['mean'].max() + 0.6)

plt.tight_layout()
plt.savefig('fig_02_bar.png', dpi=120, bbox_inches='tight')
plt.show()
print("▶ Saved: fig_02_bar.png")

### 2.3 Scatter Plot — Total Bill vs Tip (coloured by Party Size)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

sc = ax.scatter(
    df['total_bill'], df['tip'],
    c=df['size'], cmap='Blues', s=60,
    alpha=0.75, edgecolors='grey', linewidths=0.4
)

# Best-fit line
m, b = np.polyfit(df['total_bill'], df['tip'], 1)
x_line = np.linspace(df['total_bill'].min(), df['total_bill'].max(), 200)
ax.plot(x_line, m*x_line + b, color='#FF5722', lw=1.8, label=f'Trend (slope={m:.2f})')

plt.colorbar(sc, ax=ax, label='Party Size')
ax.set_title('Total Bill vs Tip — coloured by Party Size', fontweight='bold')
ax.set_xlabel('Total Bill ($)')
ax.set_ylabel('Tip ($)')
ax.legend(frameon=False)

plt.tight_layout()
plt.savefig('fig_03_scatter.png', dpi=120, bbox_inches='tight')
plt.show()
print("▶ Saved: fig_03_scatter.png")

### 2.4 Line Chart — Average Tip % Across Party Sizes

In [ ]:
line_data = df.groupby('size')['tip_pct'].agg(['mean', 'std']).reset_index()

fig, ax = plt.subplots(figsize=(8, 4))

ax.plot(line_data['size'], line_data['mean'], marker='o', color='#2196F3',
        lw=2, markersize=7, label='Mean tip %')
ax.fill_between(
    line_data['size'],
    line_data['mean'] - line_data['std'],
    line_data['mean'] + line_data['std'],
    alpha=0.15, color='#2196F3', label='±1 SD'
)

for _, row in line_data.iterrows():
    ax.annotate(f"{row['mean']:.1f}%",
                xy=(row['size'], row['mean']),
                xytext=(0, 8), textcoords='offset points',
                ha='center', fontsize=9, color='#2196F3')

ax.set_title('Average Tip % by Party Size (with ±1 SD band)', fontweight='bold')
ax.set_xlabel('Party Size (people)')
ax.set_ylabel('Tip Percentage (%)')
ax.set_xticks(line_data['size'])
ax.legend(frameon=False)

plt.tight_layout()
plt.savefig('fig_04_line.png', dpi=120, bbox_inches='tight')
plt.show()
print("▶ Saved: fig_04_line.png")

---
## 3. Strategy B — Seaborn (Statistical, Declarative)

**Design rationale:** Seaborn abstracts repetitive plotting logic and integrates statistical summaries directly.  
It is best suited for rapid exploratory analysis where confidence intervals, regression lines, and faceting add value without extra code.

### 3.1 KDE + Rug Plot — Distribution of Tip Percentage (Smoker vs Non-Smoker)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

sns.kdeplot(
    data=df, x='tip_pct', hue='smoker',
    fill=True, alpha=0.35, linewidth=2,
    palette={'Yes': '#FF5722', 'No': '#2196F3'}, ax=ax
)
sns.rugplot(
    data=df, x='tip_pct', hue='smoker',
    palette={'Yes': '#FF5722', 'No': '#2196F3'},
    height=0.04, alpha=0.5, ax=ax
)

ax.set_title('Tip % Distribution by Smoker Status (KDE + Rug)', fontweight='bold')
ax.set_xlabel('Tip Percentage (%)')
ax.set_ylabel('Density')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('fig_05_kde.png', dpi=120, bbox_inches='tight')
plt.show()
print("▶ Saved: fig_05_kde.png")

### 3.2 Box Plot — Tip Distribution by Day and Time

In [ ]:
day_order = ['Thur', 'Fri', 'Sat', 'Sun']

fig, ax = plt.subplots(figsize=(9, 5))

sns.boxplot(
    data=df, x='day', y='tip', hue='time',
    order=day_order,
    palette={'Lunch': '#81D4FA', 'Dinner': '#0D47A1'},
    width=0.55, linewidth=1.2,
    flierprops=dict(marker='o', markersize=4, alpha=0.5),
    ax=ax
)

ax.set_title('Tip Distribution by Day and Meal Time (Box Plot)', fontweight='bold')
ax.set_xlabel('Day of Week')
ax.set_ylabel('Tip ($)')
ax.legend(title='Meal', frameon=False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('fig_06_boxplot.png', dpi=120, bbox_inches='tight')
plt.show()
print("▶ Saved: fig_06_boxplot.png")

### 3.3 Regression Scatter Plot — Bill vs Tip by Gender

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

sns.scatterplot(
    data=df, x='total_bill', y='tip',
    hue='sex', style='smoker',
    palette={'Male': '#1976D2', 'Female': '#E91E63'},
    s=65, alpha=0.75, ax=ax
)

for gender, color in [('Male','#1976D2'), ('Female','#E91E63')]:
    subset = df[df['sex'] == gender]
    m, b   = np.polyfit(subset['total_bill'], subset['tip'], 1)
    x_r    = np.linspace(subset['total_bill'].min(), subset['total_bill'].max(), 200)
    ax.plot(x_r, m*x_r + b, color=color, lw=2, ls='--', alpha=0.8)

ax.set_title('Total Bill vs Tip by Gender (with Trend Lines)', fontweight='bold')
ax.set_xlabel('Total Bill ($)')
ax.set_ylabel('Tip ($)')
ax.legend(title='Gender / Smoker', frameon=False, fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('fig_07_regscatter.png', dpi=120, bbox_inches='tight')
plt.show()
print("▶ Saved: fig_07_regscatter.png")

### 3.4 Heatmap — Correlation Matrix of Numeric Features

In [ ]:
numeric_cols = ['total_bill', 'tip', 'size', 'tip_pct',
                'bill_per_person', 'tip_vs_day_avg',
                'is_weekend', 'sex_encoded', 'smoker_encoded', 'time_encoded']

corr = df[numeric_cols].corr().round(2)

fig, ax = plt.subplots(figsize=(10, 7))

mask = np.triu(np.ones_like(corr, dtype=bool))   # upper triangle masked

sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.5,
    annot_kws={'size': 8}, ax=ax
)

ax.set_title('Correlation Matrix — Numeric Features (lower triangle)', fontweight='bold')

plt.tight_layout()
plt.savefig('fig_08_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()
print("▶ Saved: fig_08_heatmap.png")

### 3.5 Count Plot — Tip Generosity by Gender and Smoker Status

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cat_order = ['Low (<10%)', 'Average (10–20%)', 'Generous (>20%)']

sns.countplot(
    data=df, x='tip_category', hue='sex',
    order=cat_order,
    palette={'Male': '#1976D2', 'Female': '#E91E63'},
    ax=axes[0]
)
axes[0].set_title('Tip Category by Gender', fontweight='bold')
axes[0].set_xlabel('Tip Category')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=10)
axes[0].legend(title='Gender', frameon=False)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

sns.countplot(
    data=df, x='tip_category', hue='smoker',
    order=cat_order,
    palette={'Yes': '#FF5722', 'No': '#4CAF50'},
    ax=axes[1]
)
axes[1].set_title('Tip Category by Smoker Status', fontweight='bold')
axes[1].set_xlabel('Tip Category')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=10)
axes[1].legend(title='Smoker', frameon=False)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('fig_09_countplot.png', dpi=120, bbox_inches='tight')
plt.show()
print("▶ Saved: fig_09_countplot.png")

---
## 4. Strategy Comparison Dashboard

A combined figure placing a Matplotlib histogram and a Seaborn KDE side-by-side  
to directly compare the two strategies on the same variable (`tip_pct`).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Strategy A: Matplotlib histogram ──────────────────────────────────────────
axes[0].hist(
    df[df['smoker'] == 'No']['tip_pct'],
    bins=20, color='#2196F3', alpha=0.7, edgecolor='white', label='Non-smoker'
)
axes[0].hist(
    df[df['smoker'] == 'Yes']['tip_pct'],
    bins=20, color='#FF5722', alpha=0.6, edgecolor='white', label='Smoker'
)
axes[0].set_title('Strategy A — Matplotlib\nOverlapping Histogram', fontweight='bold')
axes[0].set_xlabel('Tip Percentage (%)')
axes[0].set_ylabel('Frequency')
axes[0].legend(frameon=False)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)
axes[0].text(0.98, 0.96,
             '+ full control\n− verbose code\n− no built-in CI',
             transform=axes[0].transAxes, ha='right', va='top',
             fontsize=8.5, color='#555',
             bbox=dict(boxstyle='round,pad=0.3', fc='#f5f5f5', ec='#ccc'))

# ── Strategy B: Seaborn KDE ───────────────────────────────────────────────────
sns.kdeplot(
    data=df, x='tip_pct', hue='smoker',
    fill=True, alpha=0.35, linewidth=2,
    palette={'Yes': '#FF5722', 'No': '#2196F3'}, ax=axes[1]
)
axes[1].set_title('Strategy B — Seaborn\nKDE (smoothed distribution)', fontweight='bold')
axes[1].set_xlabel('Tip Percentage (%)')
axes[1].set_ylabel('Density')
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)
axes[1].text(0.98, 0.96,
             '+ shows shape clearly\n+ concise code\n− hides raw counts',
             transform=axes[1].transAxes, ha='right', va='top',
             fontsize=8.5, color='#555',
             bbox=dict(boxstyle='round,pad=0.3', fc='#f5f5f5', ec='#ccc'))

fig.suptitle('Strategy Comparison — Same Data, Two Approaches', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_10_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print("▶ Saved: fig_10_comparison.png")

---
## 5. Design Justification

### 5.1 Chart-type Rationale

| Figure | Chart Type | Justification |
|--------|------------|---------------|
| Fig 1  | Histogram  | Best for single continuous variable distribution; bins reveal skew |
| Fig 2  | Bar        | Comparing discrete categories (days); error bars add statistical honesty |
| Fig 3  | Scatter    | Shows relationship between two continuous variables; colour adds a 3rd dimension |
| Fig 4  | Line       | Trend across an ordered discrete variable (party size); shade shows spread |
| Fig 5  | KDE + Rug  | Better shows distribution shape vs histogram when comparing two groups |
| Fig 6  | Box Plot   | Displays median, IQR, and outliers simultaneously; ideal for group comparison |
| Fig 7  | Scatter    | Relationship visualisation with hue + style = 4 variables in one chart |
| Fig 8  | Heatmap    | Encodes correlation matrix compactly; diverging colour scale centred at 0 |
| Fig 9  | Count Plot | Categorical frequency with subgroups; side-by-side bars avoid overplotting |
| Fig 10 | Comparison | Same data rendered two ways to directly evaluate strategy trade-offs |

### 5.2 Strategy Comparison Summary

| Dimension         | Matplotlib (Strategy A)         | Seaborn (Strategy B)              |
|-------------------|---------------------------------|-----------------------------------|
| Code verbosity    | High — every element manual     | Low — declarative API             |
| Customisability   | Very high                       | Moderate (access via `ax`)        |
| Statistical built-ins | None                       | CI, regression, KDE, FacetGrid    |
| Best use case     | Custom layouts, publication     | Rapid EDA, group comparisons      |
| Learning curve    | Steeper                         | Gentler for common chart types    |

**Recommendation:** Use Seaborn for fast exploratory analysis; switch to Matplotlib when a specific layout or annotation is needed. In practice, combining both (as in Fig 7) yields the best results.

---
## 6. Insight Analysis Report

### Finding 1 — Tip amount scales with the bill (Fig 3, Fig 7)
A clear positive linear relationship exists between `total_bill` and `tip` (slope ≈ 0.10–0.11).  
Both genders exhibit similar slopes, meaning the *rate* of tipping does not differ meaningfully by gender.  
**Implication:** Larger tables that spend more will generate higher absolute tips, regardless of who is at the table.

---

### Finding 2 — Total bill is right-skewed; most bills cluster between $10–20 (Fig 1)
The mean ($19.79) exceeds the median ($17.80), confirming right skew driven by a small number of large bills.  
**Implication:** Tip revenue is concentrated in a minority of high-spending tables — identifying these groups is key to understanding peak-revenue shifts.

---

### Finding 3 — Sunday sees the highest average tips; Friday the lowest (Fig 2)
Sunday averages ≈ $3.25, Friday ≈ $2.73.  
Weekend visits (Sat/Sun) account for the majority of observations and slightly higher per-table tips.  
**Implication:** Staffing and service quality are most critical on weekends, where tip revenue is highest.

---

### Finding 4 — Tip percentage *decreases* as party size grows (Fig 4)
Solo diners tip an average of ≈ 21 %, while parties of 6 tip ≈ 16 %.  
The SD band widens for large parties, indicating more variable tipping behaviour.  
**Implication:** Automatic gratuity policies for large parties (e.g., parties of 6+) are justified by this data.

---

### Finding 5 — Smokers tip in a wider spread but similar average (Fig 5, Fig 6)
The KDE shows smokers have a heavier right tail — some give very high tips — but the modal tip % is comparable.  
The box plot confirms non-smokers are more consistent; smokers have more outliers on both ends.  
**Implication:** Smoker status is a weak predictor of average tip but a good predictor of tip *variance*.

---

### Finding 6 — `total_bill` and `tip` are strongly correlated (r = 0.68), but `tip_pct` is not (Fig 8)
While tip dollar amount correlates strongly with the bill, tip *percentage* shows near-zero correlation with bill size.  
This means customers tip proportionally, not just a fixed dollar amount.  
`bill_per_person` correlates moderately with `tip` (r ≈ 0.50), suggesting per-head spend matters.  
**Implication:** Models predicting tip revenue should use `total_bill` (not `tip_pct`) as a feature; models predicting tipping *behaviour* should use `tip_pct` as the target.

---

### Finding 7 — Most diners fall into the 'Average (10–20%)' category; gender split is similar (Fig 9)
Roughly 60 % of all diners tip in the 10–20 % band.  
Male diners make up a larger share of 'Generous' tippers in absolute count, but that reflects the higher proportion of male diners in the dataset, not a higher rate.  
**Implication:** Tip category is driven more by table spend and occasion than by demographic factors.

---
## 7. Deliverable Summary

| Deliverable | File |
|-------------|------|
| Histogram (distribution) | `fig_01_histogram.png` |
| Bar chart (group comparison) | `fig_02_bar.png` |
| Scatter plot (relationship) | `fig_03_scatter.png` |
| Line chart (trend) | `fig_04_line.png` |
| KDE + Rug (distribution — Seaborn) | `fig_05_kde.png` |
| Box plot (group distribution) | `fig_06_boxplot.png` |
| Regression scatter (multi-variable) | `fig_07_regscatter.png` |
| Heatmap (correlation) | `fig_08_heatmap.png` |
| Count plot (categorical) | `fig_09_countplot.png` |
| Strategy comparison dashboard | `fig_10_comparison.png` |
| Design justification | Section 5 (above) |
| Insight analysis report | Section 6 (above) |
| This notebook | `sprint3_milestone3.ipynb` |

---
_End of Sprint 3 — Milestone 3_